### Mac-friendly version (no GPU needed)

This notebook uses the **OpenAI API** to generate dishes. All generation runs in the cloud, so it works on any Mac with an internet connection—no local model, no GPU, and no large downloads.

**Setup:** Set your OpenAI API key in the environment (e.g. `export OPENAI_API_KEY=sk-...`) or in a `.env` file. The notebook uses `gpt-4o-mini` by default so it stays fast and low-cost.

# Nigerian Food & Ingredients Dataset Generator
### Project Description
This project generates synthetic data on Nigerian dishes and their ingredients. You can generate dishes by region (Yoruba, Igbo, Hausa) or by category (Soups, Swallows, Snacks, Rice dishes).

### Installations

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate gradio

zsh:1: command not found: pip


### Imports

In [ ]:
# Imports
import os
import json
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gradio as grimport gc


### Logging into Huggingface

In [ ]:
# Load API key from environment (or use python-dotenv: load_dotenv() then os.getenv)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY in your environment (e.g. export OPENAI_API_KEY=sk-...)")

# Small, fast model that works great for this task and is cheap
MODEL_NAME = "gpt-4o-mini"
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

### System Prompt and User Prompt Template

In [7]:
def generate_prompt(category, number):
    category_guidance = {
        "Yoruba dishes": "Yoruba cuisine: e.g. Efo riro, Ewa agoyin, Amala, Gbegiri, Ewedu, Ila alasepo, Ofada rice, Obe ata, Moin moin, Akara",
        "Igbo dishes": "Igbo cuisine: e.g. Ofe owerri, Ofe nsala, Ukwa, Abacha, Nni ji (pounded yam), Ofe akwu, Oha soup, Egusi soup (Igbo style), Fio fio",
        "Hausa dishes": "Hausa cuisine: e.g. Tuwo shinkafa, Miyan kuka, Suya, Kilishi, Fura da nono, Dan wake, Masa, Kosai, Tuwo masara",
        "Soups": "Nigerian soups: e.g. Egusi, Ogbono, Edikang ikong, Afang, Bitterleaf, Vegetable, Okra, Efo, Draw soup (Ewedu)",
        "Swallows": "Nigerian swallows (starchy accompaniments): e.g. Pounded yam, Eba, Fufu, Semo, Amala, Tuwo shinkafa, Tuwo masara, Lafun",
        "Snacks": "Nigerian snacks: e.g. Akara, Moi moi, Puff puff, Chin chin, Suya, Kilishi, Boli, Roasted plantain, Kokoro",
        "Rice dishes": "Nigerian rice dishes: e.g. Jollof rice, Coconut rice, Fried rice, Ofada rice, Tuwo shinkafa, White rice and stew"
    }
    examples = category_guidance.get(
        category,
        "diverse Nigerian dishes from any region with authentic names and ingredients"
    )

    system_prompt = f"""You are a Nigerian cuisine expert. When asked to generate dishes, follow these rules:

1. Generate exactly {number} unique Nigerian dishes from the category: {category}
2. Never repeat any dish in your list.
3. For each dish provide:
   - The dish name (use common Nigerian names e.g. in English or local language)
   - A short list of main ingredients (5–10 items per dish).
4. Use authentic ingredient names (e.g. egusi, ogbono, palm oil, crayfish, stockfish, uziza).
5. Format each entry as: "Dish name: [name] | Ingredients: [ingredient1, ingredient2, ...]"
6. One dish per line. No numbering needed.

Examples of {category}: {examples}.

Ensure all dishes and ingredients are culturally accurate and commonly used in Nigerian cooking."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Generate {number} Nigerian dishes from the category '{category}' with their main ingredients. One dish per line in the format: Dish name: [name] | Ingredients: [list]."}
    ]

    return messages

In [ ]:
# Use 4-bit quantization only when bitsandbytes is available (e.g. Linux + CUDA).
# On macOS/Apple Silicon or when bitsandbytes isn't installed, we load the model without quantization.
try:
    import bitsandbytes
    USE_QUANTIZATION = torch.cuda.is_available()
except Exception:
    USE_QUANTIZATION = False

if USE_QUANTIZATION:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    DEVICE = "cuda"
else:
    quant_config = None
    if torch.backends.mps.is_available():
        DEVICE = "mps"  # Apple Silicon
    elif torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        DEVICE = "cpu"
    print(f"Running without quantization on {DEVICE} (bitsandbytes not used).")

Running without quantization on mps (bitsandbytes not used).


### Generation Function

In [ ]:
def generate_food_interface(category, number):
    try:
        messages = generate_prompt(category, number)
        tokenizer = AutoTokenizer.from_pretrained(PHI)
        tokenizer.pad_token = tokenizer.eos_token
        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(DEVICE)

        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=DEVICE)
        load_kwargs = {}
        if quant_config is not None:
            load_kwargs["quantization_config"] = quant_config
            load_kwargs["device_map"] = DEVICE
        model = AutoModelForCausalLM.from_pretrained(PHI, **load_kwargs)
        if quant_config is None:
            model = model.to(DEVICE)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.7,
        )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "<|assistant|>" in generated_text:
            result = generated_text.split("<|assistant|>")[-1].strip()
        else:
            result = generated_text

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return result

    except Exception as e:
        return f"Error generating dishes: {str(e)}"

### Optional: Export to JSON/CSV
Helper to parse the model output into a list of dicts for saving as JSON or CSV.

In [15]:
def parse_dishes_to_records(text):
    """Parse generated text into list of {dish_name, ingredients}."""
    records = []
    for line in text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if "|" in line and ("Dish name:" in line or "Ingredients:" in line):
            parts = line.split("|", 1)
            name_part = parts[0].replace("Dish name:", "").strip()
            ing_part = parts[1].replace("Ingredients:", "").strip() if len(parts) > 1 else ""
            ingredients = [x.strip() for x in ing_part.split(",") if x.strip()]
            records.append({"dish_name": name_part, "ingredients": ingredients})
        else:
            records.append({"dish_name": line, "ingredients": []})
    return records


def save_as_json(records, filepath="nigerian_food_ingredients.json"):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    return filepath

### Gradio Interface

In [ ]:
def create_interface():
    with gr.Blocks(title="Nigerian Food & Ingredients Generator", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🇳🇬 Nigerian Food & Ingredients Dataset Generator")
        gr.Markdown("Generate synthetic Nigerian dishes with ingredients by region or category.")

        with gr.Row():
            with gr.Column():
                category_dropdown = gr.Dropdown(
                    choices=[
                        "Yoruba dishes",
                        "Igbo dishes",
                        "Hausa dishes",
                        "Soups",
                        "Swallows",
                        "Snacks",
                        "Rice dishes"
                    ],
                    label="Category",
                    value="Soups",
                    info="Choose a food category"
                )

                number_slider = gr.Slider(
                    minimum=1,
                    maximum=20,
                    step=1,
                    value=5,
                    label="Number of dishes",
                    info="How many dishes to generate?"
                )

                generate_btn = gr.Button("Generate dishes", variant="primary", size="lg")

            with gr.Column():
                output_text = gr.Textbox(
                    label="Generated dishes & ingredients",
                    lines=18,
                    placeholder="Generated dishes will appear here...",
                    show_copy_button=True
                )

        gr.Markdown("""
        ### Categories
        - **Yoruba / Igbo / Hausa**: Dishes typical of each region.
        - **Soups**: Nigerian soups (egusi, ogbono, vegetable, etc.).
        - **Swallows**: Starchy sides (pounded yam, eba, fufu, etc.).
        - **Snacks**: Akara, puff puff, suya, etc.
        - **Rice dishes**: Jollof, coconut rice, ofada, etc.
        """)

        generate_btn.click(
            fn=generate_food_interface,
            inputs=[category_dropdown, number_slider],
            outputs=output_text
        )

        return demo


demo = create_interface()
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7872

Could not create share link. Missing file: /Users/damolaadewunmi/.cache/huggingface/gradio/frpc/frpc_darwin_arm64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_darwin_arm64
2. Rename the downloaded file to: frpc_darwin_arm64_v0.3
3. Move the file to this location: /Users/damolaadewunmi/.cache/huggingface/gradio/frpc


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/damolaadewunmi/Desktop/Work/Andela/llm_engineering/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 2768.43 MB. The target location /Users/damolaadewunmi/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct/blobs only has 136.88 MB free disk space.
  warnings.warn(


model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

/Users/damolaadewunmi/Desktop/Work/Andela/llm_engineering/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 4903.64 MB. The target location /Users/damolaadewunmi/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct/blobs only has 136.82 MB free disk space.
  warnings.warn(


model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]